# Model summary from the actual model and data

この notebook は shape を手書きしない。

- `experiment_config.py` を読む
- `../npy/X.npy` と `../npy/vocab.json` を実際に読む
- 現在の `py/models.py` から Baseline / TPL を作る
- `X.npy` の実データ1件を forward する
- forward hook で各 layer の実際の input / output shape を記録する

`experiment.ipynb` と同じ `python/` フォルダに置いて実行する。


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

import experiment_config as cfg
from py.models import Baseline, TPL
from py.tensor_projection import TensorProjection


## 1. Read the actual data

In [ ]:
X_npy = np.load(cfg.X_PATH, mmap_mode="r")

with open(cfg.VOCAB_PATH, encoding="utf-8") as f:
    vocab = json.load(f)

N, C, F, T_MAX = X_npy.shape
vocab_size = len(vocab["id_to_token"])

print("X.npy shape :", X_npy.shape)
print("samples     :", N)
print("channels    :", C)
print("n_mels      :", F)
print("max_time    :", T_MAX)
print("vocab_size  :", vocab_size)

# 実際の X.npy の第1サンプルを使う。
x = torch.from_numpy(np.array(X_npy[0:1], copy=True)).float()
print("sample shape:", tuple(x.shape))


## 2. Build the actual Baseline and TPL models

In [ ]:
c1, c2, c3 = cfg.CNN_CHANNELS

baseline = Baseline(
    F,
    vocab_size,
    c1, c2, c3,
    cfg.GRU_HIDDEN,
    cfg.GRU_LAYERS,
    cfg.DROPOUT,
)

tpl = TPL(
    F,
    T_MAX,
    vocab_size,
    c1, c2, c3,
    cfg.GRU_HIDDEN,
    cfg.GRU_LAYERS,
    cfg.DROPOUT,
)

baseline.eval()
tpl.eval()

print("Baseline parameters:", f"{sum(p.numel() for p in baseline.parameters()):,}")
print("TPL parameters     :", f"{sum(p.numel() for p in tpl.parameters()):,}")


## 3. Forward-hook summary

`Sequential` の箱そのものではなく、その中で実際に計算を行う layer を表示する。

GRU は `(output, hidden_state)` を返すため、表では次の layer に渡される `output` の shape を表示する。


In [ ]:
def tensor_shape(obj):
    if torch.is_tensor(obj):
        return tuple(obj.shape)

    if isinstance(obj, (tuple, list)):
        # GRU など: 次の layer に渡される main output を優先
        for item in obj:
            if torch.is_tensor(item):
                return tuple(item.shape)

    return None


def count_own_params(module):
    return sum(
        p.numel()
        for p in module.parameters(recurse=False)
    )


def should_hook(module):
    # Container は飛ばす
    if isinstance(module, (nn.Sequential, nn.ModuleList, nn.ModuleDict, nn.ParameterDict)):
        return False

    # TensorProjection は ParameterDict を child に持つので明示的に拾う
    if isinstance(module, TensorProjection):
        return True

    # その他は leaf module のみ
    return len(list(module.children())) == 0


def summarize_forward(model, x):
    rows = []
    handles = []

    def make_hook(name):
        def hook(module, inputs, output):
            input_shape = tensor_shape(inputs)
            output_shape = tensor_shape(output)

            # TensorProjection の trainable weights は ParameterDict 内なので
            # この layer だけ再帰的に数える
            if isinstance(module, TensorProjection):
                params = sum(p.numel() for p in module.parameters())
            else:
                params = count_own_params(module)

            rows.append({
                "Layer name": name,
                "Layer type": module.__class__.__name__,
                "Input shape": str(input_shape),
                "Output shape": str(output_shape),
                "Parameters": params,
            })
        return hook

    for name, module in model.named_modules():
        if name and should_hook(module):
            handles.append(module.register_forward_hook(make_hook(name)))

    with torch.no_grad():
        y = model(x)

    for h in handles:
        h.remove()

    summary = pd.DataFrame(rows)

    print("Model input :", tuple(x.shape))
    print("Model output:", tuple(y.shape))
    print()

    return summary


## 4. Baseline summary

In [ ]:
baseline_summary = summarize_forward(baseline, x)
display(baseline_summary)


## 5. TPL summary

In [ ]:
tpl_summary = summarize_forward(tpl, x)
display(tpl_summary)


## 6. Compact tables for writing the paper

必要な layer だけ残した版。
shape 自体は上の forward hook で取得した実測値をそのまま使う。


In [ ]:
KEEP_TYPES = {
    "Conv2d",
    "GELU",
    "MaxPool2d",
    "TensorProjection",
    "GRU",
    "Dropout",
    "Linear",
}

baseline_paper = baseline_summary[
    baseline_summary["Layer type"].isin(KEEP_TYPES)
].reset_index(drop=True)

tpl_paper = tpl_summary[
    tpl_summary["Layer type"].isin(KEEP_TYPES)
].reset_index(drop=True)

print("Baseline")
display(baseline_paper)

print("TPL")
display(tpl_paper)


## 7. LaTeX source

In [ ]:
print("=== Baseline ===")
print(
    baseline_paper.to_latex(
        index=False,
        escape=True,
        column_format="llllr",
    )
)

print("\\n=== TPL ===")
print(
    tpl_paper.to_latex(
        index=False,
        escape=True,
        column_format="llllr",
    )
)
